# Product photo → short video (LTX-Video, free)

Runtime → Change runtime type → **GPU** (T4 is the free one). Run the cells top to bottom.
The first run downloads the model (several GB), so expect a few minutes before anything renders.
Clips are about 4 seconds, 9:16 (480x832). Upscale or edit them afterwards for posting.

In [ ]:
!nvidia-smi
!pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline

pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()   # keeps memory low enough for a 16 GB card
pipe.vae.enable_tiling()
print("model ready")

In [ ]:
# Upload your product image(s) (480x832 works best; any 9:16-ish image is fine).
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
#@title Describe the motion (not the product: the photo already shows that)
image_name = list(uploaded)[0]  #@param {type:"string"}
prompt = "The camera slowly pushes in on the product. Soft light shimmers across it. Smooth, steady, commercial product video." #@param {type:"string"}
seed = 7 #@param {type:"integer"}
num_frames = 97 #@param [49, 73, 97, 121] {type:"raw"}   # must be 8n+1; 97 frames is about 4 s
steps = 30 #@param {type:"integer"}

In [ ]:
from diffusers.utils import load_image, export_to_video

image = load_image(image_name).resize((480, 832))
negative = "worst quality, inconsistent motion, blurry, jittery, distorted, warped, morphing, text, watermark"
video = pipe(
    image=image, prompt=prompt, negative_prompt=negative,
    width=480, height=832, num_frames=num_frames, num_inference_steps=steps,
    generator=torch.Generator().manual_seed(seed),
).frames[0]
out = image_name.rsplit(".", 1)[0] + "_clip.mp4"
export_to_video(video, out, fps=24)
print("saved", out)

In [ ]:
from IPython.display import Video
from google.colab import files
display(Video(out, embed=True, width=300))
files.download(out)